# 05 · Feature mart temporal y evaluación de modelos

**Pregunta.** ¿Las variables electorales históricas auditadas mejoran el baseline A+B sin fuga temporal ni deterioro fuerte en otros folds? ¿Qué valor incremental ofrece ECEG 2020 cuando realmente estaba disponible?

**Entradas.** Conteos CCPC armonizados por sexo y edad, clases A/B, componentes AMGE, baseline congelado y ZIP público ECEG 2020 de INEGI.

**Diseño.** Se agregan conteos antes de calcular tasas. Cada fold entrena sólo con elecciones objetivo anteriores. El modelo principal usa variables electorales; ECEG, publicado/modificado el 18 de agosto de 2021, queda prohibido en los backtests estrictos 2015/2018/2021 y se evalúa sólo como evidencia secundaria para 2024.

**Gate.** Mejorar el MAE ponderado 2021 de 9.38 pp, igualar su cobertura A+B y no empeorar ningún otro fold más de 0.5 pp absolutos.

**Salidas.** Feature mart, linaje de variables, predicciones, métricas, errores por entidad/clase, importancia, registro de modelos, decisión y reporte HTML.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
from IPython.display import Markdown, display

from radar27.ingest.eceg import prepare_eceg
from radar27.modeling.feature_models import (
    build_feature_mart,
    build_section_year_behavioral_features,
    evaluate_models,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs/modeling.yaml").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
display(pd.Series({"project_root": PROJECT_ROOT.name, "phase": "feature_model_evaluation"}))

## Fuente sociodemográfica pública

Se verifica tamaño, SHA-256, firma ZIP, CRC y rutas internas antes de extraer sólo la tabla seccional, el diccionario y los metadatos. La fecha de disponibilidad se conserva como una restricción de modelado, no como una nota informal.

In [ ]:
eceg_manifest, eceg_dictionary, eceg = prepare_eceg(PROJECT_ROOT)
display(eceg_manifest.T)
display(eceg_dictionary[["topic", "indicator", "mnemonic", "unit", "output_variable"]])
display(pd.Series({
    "eceg_sections": len(eceg),
    "unique_section_keys": eceg.section_key_observed.nunique(),
    "selected_source_variables": len(eceg_dictionary),
}))

In [ ]:
display(Markdown(
    "**Interpretación.** ECEG representa una fotografía censal 2020 por sección. Su fecha oficial "
    "de modificación es posterior a la elección 2021; por contrato sólo puede entrar a la prueba "
    "secundaria 2024 y a un futuro modelo 2027. No se usa como crosswalk territorial."
))

## Variables electorales por sección–año

Los 54.9 millones de registros sexo–edad se reducen a numeradores y denominadores auditables. Las tasas por sexo y edad se calcularán después de agregar cada sección o componente AMGE.

In [ ]:
section_year = build_section_year_behavioral_features(PROJECT_ROOT)
behavioral_summary = (
    section_year.groupby("election_year", as_index=False)
    .agg(
        sections=("section_key_observed", "nunique"),
        nominal_list=("nominal_list", "sum"),
        voted=("voted", "sum"),
        did_not_vote=("did_not_vote", "sum"),
    )
)
display(behavioral_summary)

In [ ]:
display(Markdown(
    f"**Resultado.** Se construyeron `{len(section_year):,}` filas sección–año. "
    "Cada componente posterior suma conteos; no promedia porcentajes seccionales."
))

## Feature mart A+B

Las unidades A conservan su sección. Las B usan el componente oficial AMGE. Cada fila contiene outcome sólo para evaluar; la matriz predictora usa exclusivamente elecciones anteriores. ECEG conserva una bandera explícita de elegibilidad temporal.

In [ ]:
feature_mart, feature_coverage = build_feature_mart(PROJECT_ROOT)
display(feature_coverage)
display(pd.read_csv(PROJECT_ROOT / "artifacts/tables/model_feature_lineage.csv"))

In [ ]:
mart_checks = {
    "unit_target_unique": not feature_mart.duplicated(["target_year", "evaluation_unit_id"]).any(),
    "electoral_features_strictly_prior": bool(
        (feature_mart.max_electoral_feature_year < feature_mart.target_year).all()
    ),
    "eceg_forbidden_in_2021_strict_backtest": bool(
        (~feature_mart.loc[feature_mart.target_year.eq(2021), "eceg_strict_backtest_allowed"]).all()
    ),
    "targets_defined_only_from_target_counts": bool(
        feature_mart.actual.dropna().between(0, 1).all()
    ),
}
display(pd.Series(mart_checks, name="aprobado").to_frame())
assert all(mart_checks.values())

## Backtest de modelos

Ridge sirve como referencia interpretable. El boosting principal aprende una corrección residual conservadora de 25% sobre el rezago de la misma familia; si falta ese rezago usa una predicción directa. El factor se fijó usando el fold 2018, anterior al gate principal 2021. Imputación, escalamiento y categorías se ajustan sólo con targets anteriores.

In [ ]:
model_metrics, model_predictions, feature_importance, model_decision = evaluate_models(
    PROJECT_ROOT, feature_mart
)
display(model_metrics)
display(pd.Series(model_decision).to_frame("valor"))

In [ ]:
primary = model_metrics[model_metrics.target_year.eq(2021)].copy()
primary["weighted_mae_pp"] = 100 * primary["identified_weighted_mae"]
primary["coverage_pct"] = 100 * primary["section_coverage"]
display(primary[["model_id", "weighted_mae_pp", "coverage_pct", "bias", "share_within_5pp"]])

selected = model_decision["selected_for_next_phase"]
status = "APROBADO" if model_decision["candidate_approved"] else "NO APROBADO"
display(Markdown(
    f"**Gate principal: {status}.** Selección: `{selected}`. "
    f"Baseline congelado: `{100*model_decision['frozen_baseline_identified_weighted_mae']:.2f}` pp; "
    f"mejor candidato: `{100*model_decision['best_candidate_identified_weighted_mae']:.2f}` pp."
))

## Robustez, procedencia e interpretación

Se muestran los controles de deterioro y las variables más influyentes. La importancia por permutación es descriptiva sobre una muestra determinista y no implica causalidad.

In [ ]:
display(pd.DataFrame(model_decision["robustness_checks"]))
top_importance = (
    feature_importance[
        feature_importance.model_id.eq("hist_gradient_boosting_electoral")
        & feature_importance.target_year.eq(2021)
    ]
    .sort_values("importance", ascending=False)
    .head(15)
)
display(top_importance)
display(pd.read_csv(PROJECT_ROOT / "artifacts/tables/model_run_registry.csv"))

In [ ]:
prediction_checks = {
    "emitted_predictions_are_probabilities": bool(
        model_predictions.prediction.dropna().between(0, 1).all()
    ),
    "eceg_models_only_evaluate_2024": set(
        model_metrics.loc[model_metrics.model_id.str.endswith("_eceg"), "target_year"]
    ) <= {2024},
    "training_years_are_strictly_prior": all(
        all(int(year) < int(row.target_year) for year in str(row.training_target_years).split("|"))
        for row in model_metrics.itertuples(index=False)
        if str(row.training_target_years)
    ),
}
display(pd.Series(prediction_checks, name="aprobado").to_frame())
assert all(prediction_checks.values())

## Reporte final

El reporte reúne fuente, cobertura, métricas, decisión, robustez, importancia y registro de ejecución. Las tablas detalladas permanecen como artefactos máquina-legibles.

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/feature_model_evaluation.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join([
    "<h1>RADAR 27 · Feature mart y evaluación temporal</h1>",
    "<p>Generado por notebooks/05_feature_model_evaluation.ipynb.</p>",
    "<h2>Fuente ECEG</h2>", eceg_manifest.to_html(index=False),
    "<h2>Cobertura del feature mart</h2>", feature_coverage.to_html(index=False),
    "<h2>Métricas</h2>", model_metrics.to_html(index=False),
    "<h2>Decisión</h2><pre>" + json.dumps(model_decision, ensure_ascii=False, indent=2) + "</pre>",
    "<h2>Robustez</h2>", pd.DataFrame(model_decision["robustness_checks"]).to_html(index=False),
    "<h2>Importancia principal 2021</h2>", top_importance.to_html(index=False),
]), encoding="utf-8")

required_outputs = [
    PROJECT_ROOT / "data/manifests/eceg_source_manifest.csv",
    PROJECT_ROOT / "artifacts/tables/eceg_2020_section_features.parquet",
    PROJECT_ROOT / "artifacts/tables/section_year_behavioral_features.parquet",
    PROJECT_ROOT / "artifacts/tables/feature_mart_ab.parquet",
    PROJECT_ROOT / "artifacts/tables/model_feature_lineage.csv",
    PROJECT_ROOT / "artifacts/tables/model_backtest_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/model_backtest_metrics.csv",
    PROJECT_ROOT / "artifacts/tables/model_errors_by_entity_class.csv",
    PROJECT_ROOT / "artifacts/tables/model_feature_importance.csv",
    PROJECT_ROOT / "artifacts/tables/model_run_registry.csv",
    PROJECT_ROOT / "artifacts/model_decision.json",
    report_path,
]
final_checks = {
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    "source_hash_verified": eceg_manifest.validation_status.eq(
        "zip_signature_crc_paths_hash_verified"
    ).all(),
    "decision_is_explicit": selected in {
        "hist_gradient_boosting_electoral", "stable_plus_lineage_baseline"
    },
    **mart_checks,
    **prediction_checks,
}
display(pd.Series(final_checks, name="aprobado").to_frame())
assert all(final_checks.values())

In [ ]:
display(Markdown(
    f"**Cierre.** `{sum(final_checks.values())}/{len(final_checks)}` controles pasaron. "
    f"La decisión quedó congelada en `artifacts/model_decision.json`. "
    f"Tiempo total: `{time.perf_counter()-RUN_STARTED:.1f}` s."
))